In [ ]:
from __future__ import annotations
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print(project_root)
import json
import torch

from src.train_utils import load_tokenizer
from src.data_processor import IronCellCollator
from src.data_processor.data_processor_adaptive import AdaptiveIronCellCollator, AdaptiveZipperBuilder

from src.data_processor.fixed import IronCellCollator, ZipperBuilder



class _Args:
    def __init__(self):
        self.model_name = ""
        self.phase = "phase2"
        self.resume_path = None
        self.load_weights_only = True
        self.parallel = "none"


def fixed_chunk_lens_from_text(tokenizer, text: str, chunk_size: int = 16):
    ids = tokenizer.encode(text, add_special_tokens=False)
    n = len(ids)
    return [min(chunk_size, n - i) for i in range(0, n, chunk_size)]


def decode_chunks(tokenizer, text: str, chunk_lens):
    ids = tokenizer.encode(text, add_special_tokens=False)
    assert sum(chunk_lens) == len(ids), (
        f"sum(chunk_lens)={sum(chunk_lens)} != raw_len={len(ids)}"
    )
    out = []
    cur = 0
    for i, ln in enumerate(chunk_lens):
        chunk_ids = ids[cur:cur+ln]
        out.append({
            "chunk_idx": i,
            "len": ln,
            "ids": chunk_ids,
            "tokens": tokenizer.convert_ids_to_tokens(chunk_ids),
            "text": tokenizer.decode(chunk_ids, skip_special_tokens=False),
        })
        cur += ln
    return out


def tensor_preview(x, max_rows=40, max_cols=40):
    if not isinstance(x, torch.Tensor):
        print(x)
        return
    print("shape:", tuple(x.shape))
    if x.ndim == 1:
        print(x[:max_rows])
    elif x.ndim == 2:
        print(x[:max_rows, :max_cols])
    elif x.ndim == 3:
        print(x[:1, :max_rows, :max_cols])
    else:
        print(x)


def count_supervised(labels: torch.Tensor):
    return int((labels != -100).sum().item())


def show_batch_basic(batch, prefix=""):
    print(f"\n{prefix}zipper_input_ids")
    tensor_preview(batch.zipper_input_ids)

    print(f"\n{prefix}labels")
    tensor_preview(batch.labels)
    print("num supervised tokens:", count_supervised(batch.labels))

    print(f"\n{prefix}attention_mask_2d")
    tensor_preview(batch.attention_mask_2d)

    print(f"\n{prefix}position_ids")
    tensor_preview(batch.position_ids)

    if hasattr(batch, "chunk_input_ids"):
        print(f"\n{prefix}chunk_input_ids")
        tensor_preview(batch.chunk_input_ids)

    if hasattr(batch, "chunk_attention_mask"):
        print(f"\n{prefix}chunk_attention_mask")
        tensor_preview(batch.chunk_attention_mask)

    if hasattr(batch, "memory_positions"):
        print(f"\n{prefix}memory_positions")
        tensor_preview(batch.memory_positions)

    if hasattr(batch, "prefix_lens"):
        print(f"\n{prefix}prefix_lens")
        tensor_preview(batch.prefix_lens)

    if hasattr(batch, "valid_lens"):
        print(f"\n{prefix}valid_lens")
        tensor_preview(batch.valid_lens)

torch.set_printoptions(linewidth=300)

/default-vepfs/public/user/ga/Iron/IronMan


In [ ]:
def compare_tensor(name, a, b):
    print(f"\n=== {name} ===")
    if isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor):
        print("shape a:", tuple(a.shape))
        print("shape b:", tuple(b.shape))
        same_shape = tuple(a.shape) == tuple(b.shape)
        print("same_shape:", same_shape)
        if not same_shape:
            return

        equal = torch.equal(a, b)
        print("torch.equal:", equal)

        if not equal:
            diff = (a != b)
            num_diff = int(diff.sum().item())
            print("num_diff:", num_diff)

            # 找前几个不同位置
            idx = diff.nonzero(as_tuple=False)
            print("first diff idx:", idx[:10])

            # 打印前几个不同值
            for k in range(min(10, idx.shape[0])):
                pos = tuple(idx[k].tolist())
                print(f"pos={pos}, a={a[pos].item()}, b={b[pos].item()}")
    else:
        print("a:", a)
        print("b:", b)
        print("equal:", a == b)

def compare_builders(fixed, adaptive, tokenizer):
    print("\n================ BASIC FIELDS ================")
    print("fixed.raw_len:", fixed.raw_len)
    print("adaptive.raw_len:", adaptive.raw_len)

    print("fixed.valid_len:", fixed.valid_len)
    print("adaptive.valid_len:", adaptive.valid_len)

    print("fixed.prefix_len:", fixed.prefix_len)
    print("adaptive.prefix_len:", adaptive.prefix_len)

    # print("fixed.num_raw_segments:", fixed.num_raw_segments)
    # print("adaptive.num_raw_segments:", adaptive.num_raw_segments)

    print("fixed.num_cmp_chunks:", fixed.num_cmp_chunks)
    print("adaptive.num_cmp_chunks:", adaptive.num_cmp_chunks)

    print("fixed.memory_positions:", fixed.memory_positions)
    print("adaptive.memory_positions:", adaptive.memory_positions)

    print("\n================ RAW CHUNKS LEN ================")
    print("fixed raw chunk lens:", [len(x) for x in fixed.raw_chunks])
    print("adaptive raw chunk lens:", [len(x) for x in adaptive.raw_chunks])

    print("\n================ CMP CHUNKS LEN ================")
    print("fixed cmp chunk lens:", [len(x) for x in fixed.cmp_wrapped_chunks])
    print("adaptive cmp chunk lens:", [len(x) for x in adaptive.cmp_wrapped_chunks])

    fixed_gen = torch.tensor(fixed.gen_input_ids, dtype=torch.long)
    adaptive_gen = torch.tensor(adaptive.gen_input_ids, dtype=torch.long)

    compare_tensor("gen_input_ids", fixed_gen, adaptive_gen)

    fixed_labels = fixed.build_gen_labels(device=torch.device("cpu"))
    adaptive_labels = adaptive.build_gen_labels(device=torch.device("cpu"))
    compare_tensor("labels", fixed_labels, adaptive_labels)

    fixed_attn, fixed_pos = fixed.build_gen_attention_and_pos(
        seq_len=fixed.valid_len, device=torch.device("cpu")
    )
    adaptive_attn, adaptive_pos = adaptive.build_gen_attention_and_pos(
        seq_len=adaptive.valid_len, device=torch.device("cpu")
    )

    compare_tensor("attention_mask_2d", fixed_attn, adaptive_attn)
    compare_tensor("position_ids", fixed_pos, adaptive_pos)

    print("\n================ DECODE PREVIEW ================")
    print("fixed gen tokens:")
    print(tokenizer.convert_ids_to_tokens(fixed.gen_input_ids[:120]))
    print("\nadaptive gen tokens:")
    print(tokenizer.convert_ids_to_tokens(adaptive.gen_input_ids[:120]))
    

In [19]:
args = _Args()
args.model_name = "/default-vepfs/public/user/ga/Iron/models/Llama-3.1-8B"
tokenizer, _ = load_tokenizer(args)

# 方式 A：手写一条
text = "The secret passcode is tiger. It is very important to remember this sentence for later generation. This is only a debug example."

text = "The secret passcode is tiger. It is very important to "

# 方式 B：从你的 jsonl 读第一条
# with open("/default-vepfs/public/user/ga/Iron/IronMan/data/test_collator.jsonl", "r", encoding="utf-8") as f:
#     obj = json.loads(next(f))
# text = obj["text"]

raw_ids = tokenizer.encode(text, add_special_tokens=False)
print("raw_len:", len(raw_ids))
print("raw_tokens preview:")
print(tokenizer.convert_ids_to_tokens(raw_ids[:80]))

raw_len: 13
raw_tokens preview:
['The', 'Ġsecret', 'Ġpass', 'code', 'Ġis', 'Ġtiger', '.', 'ĠIt', 'Ġis', 'Ġvery', 'Ġimportant', 'Ġto', 'Ġ']


In [20]:
chunk_lens = fixed_chunk_lens_from_text(tokenizer, text, chunk_size=4)
print("chunk_lens:", chunk_lens)
print("sum(chunk_lens):", sum(chunk_lens), "raw_len:", len(raw_ids))

decoded = decode_chunks(tokenizer, text, chunk_lens)
for x in decoded:
    print(f"\n[Chunk {x['chunk_idx']}] len={x['len']}")
    print(x["text"])

chunk_lens: [4, 4, 4, 1]
sum(chunk_lens): 13 raw_len: 13

[Chunk 0] len=4
The secret passcode

[Chunk 1] len=4
 is tiger. It

[Chunk 2] len=4
 is very important to

[Chunk 3] len=1
 


In [21]:
builder = AdaptiveZipperBuilder(
    tokenizer=tokenizer,
    prompt=text,
    raw_chunk_lens=chunk_lens,
    buffer_size=0,
    num_v=2,   # 一定和训练保持一致
    random_gate=0,
    truncate_len=None,
)

print("raw_len:", builder.raw_len)
print("num_raw_segments:", builder.num_raw_segments)
print("num_cmp_chunks:", builder.num_cmp_chunks)
print("prefix_len:", builder.prefix_len)
print("valid_len:", builder.valid_len)
print("memory_positions:", builder.memory_positions)

print("raw chunk lens from builder:", [len(x) for x in builder.raw_chunks])
print("cmp_wrapped_chunks lens:", [len(x) for x in builder.cmp_wrapped_chunks])

assert [len(x) for x in builder.raw_chunks] == chunk_lens
assert len(builder.cmp_wrapped_chunks) == len(chunk_lens) - 1

print("\nDecoded raw chunks from builder:")
for i, c in enumerate(builder.raw_chunks):
    print(f"[raw_chunk {i}] len={len(c)}")
    print(tokenizer.decode(c, skip_special_tokens=False))

print("\nDecoded cmp_wrapped_chunks:")
for i, c in enumerate(builder.cmp_wrapped_chunks):
    print(f"[cmp_chunk {i}] len={len(c)}")
    print(tokenizer.decode(c, skip_special_tokens=False))

raw_len: 13
num_raw_segments: 4
num_cmp_chunks: 3
prefix_len: 14
valid_len: 27
memory_positions: [5, 8, 11]
raw chunk lens from builder: [4, 4, 4, 1]
cmp_wrapped_chunks lens: [4, 4, 4]

Decoded raw chunks from builder:
[raw_chunk 0] len=4
The secret passcode
[raw_chunk 1] len=4
 is tiger. It
[raw_chunk 2] len=4
 is very important to
[raw_chunk 3] len=1
 

Decoded cmp_wrapped_chunks:
[cmp_chunk 0] len=4
The secret passcode
[cmp_chunk 1] len=4
 is tiger. It
[cmp_chunk 2] len=4
 is very important to


In [22]:
labels = builder.build_gen_labels(device=torch.device("cpu"))
attn, pos = builder.build_gen_attention_and_pos(
    seq_len=builder.valid_len,
    device=torch.device("cpu"),
)

torch.set_printoptions(
    linewidth=1000,     # 最大单行字符数，调大 → 很少换行
    edgeitems=8,        # 两边各显示多少个元素（默认3）
    threshold=10000,    # 超过多少个元素才省略（...），调大就不会省略
    precision=4,        # 小数点后保留几位
    sci_mode=False      # 不要用科学计数法（可选）
)
print("labels:")
tensor_preview(labels)
print("num supervised tokens:", int((labels != -100).sum().item()))

print("\nattention mask:")
tensor_preview(attn)

print("\nposition ids:")
tensor_preview(pos)

print("\nDecoded full gen_input_ids:")
print(tokenizer.convert_ids_to_tokens(builder.gen_input_ids[:120]))

labels:
shape: (27,)
tensor([ -100,  -100,  -100,  -100,  -100,   791,  -100,  -100,   374,  -100,  -100,   374,  -100,  -100,   220,  6367,  1522,  1889,  -100, 52835,    13,  1102,  -100,  1633,  3062,   311,  -100])
num supervised tokens: 13

attention mask:
shape: (27, 27)
tensor([[ True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False],
        [ True,  True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False],
        [ True,  True,  True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False],
        [ True,  True,  True,  True, False, False, False, False, False, False, False, False, False, False, False, False, Fals

In [35]:
def compare_tensor(name, a, b):
    print(f"\n=== {name} ===")
    if isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor):
        print("shape a:", tuple(a.shape))
        print("shape b:", tuple(b.shape))
        same_shape = tuple(a.shape) == tuple(b.shape)
        print("same_shape:", same_shape)
        if not same_shape:
            return

        equal = torch.equal(a, b)
        print("torch.equal:", equal)

        if not equal:
            diff = (a != b)
            num_diff = int(diff.sum().item())
            print("num_diff:", num_diff)

            # 找前几个不同位置
            idx = diff.nonzero(as_tuple=False)
            print("first diff idx:", idx[:10])

            # 打印前几个不同值
            for k in range(min(10, idx.shape[0])):
                pos = tuple(idx[k].tolist())
                print(f"pos={pos}, a={a[pos].item()}, b={b[pos].item()}")
    else:
        print("a:", a)
        print("b:", b)
        print("equal:", a == b)

def compare_builders(fixed, adaptive, tokenizer):
    print("\n================ BASIC FIELDS ================")
    print("fixed.raw_len:", fixed.raw_len)
    print("adaptive.raw_len:", adaptive.raw_len)

    print("fixed.valid_len:", fixed.valid_len)
    print("adaptive.valid_len:", adaptive.valid_len)

    print("fixed.prefix_len:", fixed.prefix_len)
    print("adaptive.prefix_len:", adaptive.prefix_len)

    # print("fixed.num_raw_segments:", fixed.num_raw_segments)
    # print("adaptive.num_raw_segments:", adaptive.num_raw_segments)

    print("fixed.num_cmp_chunks:", fixed.num_cmp_chunks)
    print("adaptive.num_cmp_chunks:", adaptive.num_cmp_chunks)

    print("fixed.memory_positions:", fixed.memory_positions)
    print("adaptive.memory_positions:", adaptive.memory_positions)

    # print("\n================ RAW CHUNKS LEN ================")
    # print("fixed raw chunk lens:", [len(x) for x in fixed.raw_chunks])
    # print("adaptive raw chunk lens:", [len(x) for x in adaptive.raw_chunks])

    # print("\n================ CMP CHUNKS LEN ================")
    # print("fixed cmp chunk lens:", [len(x) for x in fixed.cmp_wrapped_chunks])
    # print("adaptive cmp chunk lens:", [len(x) for x in adaptive.cmp_wrapped_chunks])

    fixed_gen = torch.tensor(fixed.gen_input_ids, dtype=torch.long)
    adaptive_gen = torch.tensor(adaptive.gen_input_ids, dtype=torch.long)

    compare_tensor("gen_input_ids", fixed_gen, adaptive_gen)

    fixed_labels = fixed.build_gen_labels(device=torch.device("cpu"))
    adaptive_labels = adaptive.build_gen_labels(device=torch.device("cpu"))
    compare_tensor("labels", fixed_labels, adaptive_labels)

    fixed_attn, fixed_pos = fixed.build_gen_attention_and_pos(
        seq_len=fixed.valid_len, device=torch.device("cpu")
    )
    adaptive_attn, adaptive_pos = adaptive.build_gen_attention_and_pos(
        seq_len=adaptive.valid_len, device=torch.device("cpu")
    )

    compare_tensor("attention_mask_2d", fixed_attn, adaptive_attn)
    compare_tensor("position_ids", fixed_pos, adaptive_pos)

    print("\n================ DECODE PREVIEW ================")
    print("fixed gen tokens:")
    print(tokenizer.convert_ids_to_tokens(fixed.gen_input_ids[:120]))
    print("\nadaptive gen tokens:")
    print(tokenizer.convert_ids_to_tokens(adaptive.gen_input_ids[:120]))
    

In [33]:
fixed = ZipperBuilder(
    tokenizer=tokenizer,
    prompt=text,
    chunk_size=4,
    buffer_size=0,
    num_v=2,   # 一定和训练保持一致
    random_gate=0,
    truncate_len=None,
)

labels = fixed.build_gen_labels(device=torch.device("cpu"))
attn, pos = fixed.build_gen_attention_and_pos(
    seq_len=fixed.valid_len,
    device=torch.device("cpu"),
)

torch.set_printoptions(
    linewidth=1000,     # 最大单行字符数，调大 → 很少换行
    edgeitems=8,        # 两边各显示多少个元素（默认3）
    threshold=10000,    # 超过多少个元素才省略（...），调大就不会省略
    precision=4,        # 小数点后保留几位
    sci_mode=False      # 不要用科学计数法（可选）
)
print("labels:")
tensor_preview(labels)
print("num supervised tokens:", int((labels != -100).sum().item()))

print("\nattention mask:")
tensor_preview(attn)

print("\nposition ids:")
tensor_preview(pos)

print("\nDecoded full gen_input_ids:")
print(tokenizer.convert_ids_to_tokens(builder.gen_input_ids[:120]))

labels:
shape: (27,)
tensor([ -100,  -100,  -100,  -100,  -100,   791,  -100,  -100,   374,  -100,  -100,   374,  -100,  -100,   220,  6367,  1522,  1889,  -100, 52835,    13,  1102,  -100,  1633,  3062,   311,  -100])
num supervised tokens: 13

attention mask:
shape: (27, 27)
tensor([[ True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False],
        [ True,  True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False],
        [ True,  True,  True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False],
        [ True,  True,  True,  True, False, False, False, False, False, False, False, False, False, False, False, False, Fals

In [36]:
# 假设 chunk_lens 已经是 [16,16,...] 这种
chunk_lens = fixed_chunk_lens_from_text(tokenizer, text, chunk_size=4)

fixed = ZipperBuilder(
    tokenizer=tokenizer,
    prompt=text,
    chunk_size=4,      # 一定和 chunk_lens 对齐
    buffer_size=0,
    num_v=2,
    random_gate=0,
    truncate_len=None,
)

adaptive = AdaptiveZipperBuilder(
    tokenizer=tokenizer,
    prompt=text,
    raw_chunk_lens=chunk_lens,
    buffer_size=0,
    num_v=2,
    random_gate=0,
    truncate_len=None,
)

compare_builders(fixed, adaptive, tokenizer)


================ BASIC FIELDS ================
fixed.raw_len: 13
adaptive.raw_len: 13
fixed.valid_len: 27
adaptive.valid_len: 27
fixed.prefix_len: 14
adaptive.prefix_len: 14
fixed.num_cmp_chunks: 3
adaptive.num_cmp_chunks: 3
fixed.memory_positions: [5, 8, 11]
adaptive.memory_positions: [5, 8, 11]

=== gen_input_ids ===
shape a: (27,)
shape b: (27,)
same_shape: True
torch.equal: True

=== labels ===
shape a: (27,)
shape b: (27,)
same_shape: True
torch.equal: True

=== attention_mask_2d ===
shape a: (27, 27)
shape b: (27, 27)
same_shape: True
torch.equal: True

=== position_ids ===
shape a: (27,)
shape b: (27,)
same_shape: True
torch.equal: True

================ DECODE PREVIEW ================
fixed gen tokens:
['<|begin_of_text|>', '<soc>', '<v_none>', '<v_none>', '<eoc>', '<v_none>', '<v_none>', '<eoc>', '<v_none>', '<v_none>', '<eoc>', '<v_none>', '<v_none>', '<eoc>', 'The', 'Ġsecret', 'Ġpass', 'code', 'Ġis', 'Ġtiger', '.', 'ĠIt', 'Ġis', 'Ġvery', 'Ġimportant', 'Ġto', 'Ġ']

adaptive 

In [40]:
import json
import torch




class _Args:
    def __init__(self):
        self.model_name = "/default-vepfs/public/user/ga/Iron/models/Llama-3.1-8B"
        self.phase = "phase2"
        self.resume_path = None
        self.load_weights_only = True
        self.parallel = "none"


def load_fixed_items(path, max_samples=8):
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= max_samples:
                break
            obj = json.loads(line)
            items.append((obj["text"], i))
    return items


def load_adaptive_items(path, max_samples=8):
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= max_samples:
                break
            obj = json.loads(line)
            items.append({
                "text": obj["text"],
                "idx": i,
                "chunk_lens": obj["chunk_lens"],
            })
    return items


def compare_tensor(name, a, b):
    print(f"\n=== {name} ===")
    print("shape a:", tuple(a.shape))
    print("shape b:", tuple(b.shape))
    same_shape = tuple(a.shape) == tuple(b.shape)
    print("same_shape:", same_shape)
    if not same_shape:
        return False

    equal = torch.equal(a, b)
    print("torch.equal:", equal)
    if not equal:
        diff = (a != b)
        num_diff = int(diff.sum().item())
        print("num_diff:", num_diff)
        idx = diff.nonzero(as_tuple=False)
        print("first diff idx:", idx[:10])
        for k in range(min(10, idx.shape[0])):
            pos = tuple(idx[k].tolist())
            print(f"pos={pos}, a={a[pos].item()}, b={b[pos].item()}")
    return equal

In [41]:
args = _Args()
tokenizer, _ = load_tokenizer(args)

fixed_collator = IronCellCollator(
    tokenizer,
    chunk_size=16,
    num_v=2,
    buffer_size=0,
    random_gate=0,
)

adaptive_collator = AdaptiveIronCellCollator(
    tokenizer,
    num_v=2,
    buffer_size=0,
    random_gate=0,
)

In [46]:
fixed_items = load_fixed_items("/default-vepfs/public/user/ga/Iron/IronMan/data/eval_fixed.jsonl", max_samples=3)
adaptive_items = load_adaptive_items("/default-vepfs/public/user/ga/Iron/IronMan/data/eval_fixed.jsonl", max_samples=3)

# 先确认 text 顺序一致
for i, (fx, ad) in enumerate(zip(fixed_items, adaptive_items)):
    assert fx[0] == ad["text"], f"text mismatch at sample {i}"

fixed_batch = fixed_collator(fixed_items)
adaptive_batch = adaptive_collator(adaptive_items)

In [47]:
compare_tensor("zipper_input_ids",
               fixed_batch.zipper_input_ids,
               adaptive_batch.zipper_input_ids)

compare_tensor("labels",
               fixed_batch.labels,
               adaptive_batch.labels)

compare_tensor("attention_mask_2d",
               fixed_batch.attention_mask_2d,
               adaptive_batch.attention_mask_2d)

compare_tensor("position_ids",
               fixed_batch.position_ids,
               adaptive_batch.position_ids)

compare_tensor("chunk_input_ids",
               fixed_batch.chunk_input_ids,
               adaptive_batch.chunk_input_ids)

compare_tensor("chunk_attention_mask",
               fixed_batch.chunk_attention_mask,
               adaptive_batch.chunk_attention_mask)

compare_tensor("memory_positions",
               fixed_batch.memory_positions,
               adaptive_batch.memory_positions)

compare_tensor("prefix_lens",
               fixed_batch.prefix_lens,
               adaptive_batch.prefix_lens)

compare_tensor("valid_lens",
               fixed_batch.valid_lens,
               adaptive_batch.valid_lens)

print("fixed supervised:", int((fixed_batch.labels != -100).sum().item()))
print("adaptive supervised:", int((adaptive_batch.labels != -100).sum().item()))


=== zipper_input_ids ===
shape a: (3, 4464)
shape b: (3, 4464)
same_shape: True
torch.equal: True

=== labels ===
shape a: (3, 4464)
shape b: (3, 4464)
same_shape: True
torch.equal: True

=== attention_mask_2d ===
shape a: (3, 4464, 4464)
shape b: (3, 4464, 4464)
same_shape: True
torch.equal: True

=== position_ids ===
shape a: (3, 4464)
shape b: (3, 4464)
same_shape: True
torch.equal: True

=== chunk_input_ids ===
shape a: (3, 234, 16)
shape b: (3, 234, 16)
same_shape: True
torch.equal: True

=== chunk_attention_mask ===
shape a: (3, 234, 16)
shape b: (3, 234, 16)
same_shape: True
torch.equal: True

=== memory_positions ===
shape a: (3, 234)
shape b: (3, 234)
same_shape: True
torch.equal: True

=== prefix_lens ===
shape a: (3,)
shape b: (3,)
same_shape: True
torch.equal: True

=== valid_lens ===
shape a: (3,)
shape b: (3,)
same_shape: True
torch.equal: True
fixed supervised: 10320
adaptive supervised: 10320


In [62]:
from src.attention import TrainStepModule
from src.train_utils import load_model

run_args = _Args()
run_args.phase = "phase2"
run_args.resume_path = "/default-vepfs/public/user/ga/Iron/checkpoints/phase-full-dynamicq/phase-full_step_480" 
run_args.load_weights_only = True
run_args.parallel = "none"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer, is_resume = load_tokenizer(run_args)
print(run_args.resume_path)
print(is_resume)
model = load_model(run_args, tokenizer, device, is_resume=is_resume)
model.eval()

step_module = TrainStepModule(model, phase="phase2")

with torch.no_grad():
    with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=(device.type=="cuda")):
        fixed_loss, _, _ = step_module(fixed_batch)
        adaptive_loss, _, _ = step_module(adaptive_batch)

print("fixed_loss:", float(fixed_loss))
print("adaptive_loss:", float(adaptive_loss))

The tokenizer you are loading from '/default-vepfs/public/user/ga/Iron/checkpoints/phase-full-dynamicq/phase-full_step_480' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


/default-vepfs/public/user/ga/Iron/checkpoints/phase-full-dynamicq/phase-full_step_480
True


Loading checkpoint shards: 100%|██████████| 7/7 [00:00<00:00, 35.45it/s]


fixed_loss: 6.209934711456299
adaptive_loss: 6.209934711456299


In [60]:
fixed_items = load_fixed_items("/default-vepfs/public/user/ga/Iron/IronMan/data/eval_fixed.jsonl", max_samples=1)
adaptive_items = load_adaptive_items("/default-vepfs/public/user/ga/Iron/IronMan/data/eval_fixed.jsonl", max_samples=1)

# 先确认 text 顺序一致
for i, (fx, ad) in enumerate(zip(fixed_items, adaptive_items)):
    assert fx[0] == ad["text"], f"text mismatch at sample {i}"

fixed_batch = fixed_collator(fixed_items)
adaptive_batch = adaptive_collator(adaptive_items)
with torch.no_grad():
    with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=(device.type=="cuda")):
        fixed_loss, _, _ = step_module(fixed_batch)
        adaptive_loss, _, _ = step_module(adaptive_batch)

print("fixed_loss:", float(fixed_loss))
print("adaptive_loss:", float(adaptive_loss))

fixed_loss: 6.209934711456299
adaptive_loss: 6.209934711456299
